# 🟢 Interview: NTK-aware RoPE Scaling

Interview solution for NTK-aware Rotary Position Embedding scaling.

$$\text{base}_{\text{new}} = 10000 \cdot \text{scale}^{D/(D-2)}$$

In [ ]:
import torch


In [ ]:
# ✅ INTERVIEW

def ntk_rope(q, k, scale):
    """
    手写 NTK-aware RoPE —— 面试版。

    面试考点:
    1. RoPE 的本质：把每对相邻维度当作 2D 向量做旋转
    2. NTK-aware 只改 base，不改旋转公式本身
    3. base_new = 10000 * scale^(D/(D-2))，指数 D/(D-2) 是关键
    4. 旋转公式: x1' = x1*cos - x2*sin, x2' = x1*sin + x2*cos

    参数: q, k: (B, S, D), scale: float
    返回: (q_rotated, k_rotated), 各 (B, S, D)
    """
    B, S, D = q.shape

    # ---- Step 1: NTK-aware 基底 ----
    # 标准 RoPE base=10000, NTK-aware 通过 scale 调整
    # D/(D-2) 这个指数确保频率分布的「重心」恰好移动到新上下文长度
    new_base = 10000.0 * (scale ** (D / (D - 2)))

    # ---- Step 2: 频率 ----
    pos = torch.arange(S, device=q.device).float().unsqueeze(1)  # (S, 1)
    dim = torch.arange(0, D, 2, device=q.device).float()          # (D/2,)
    freqs = 1.0 / (new_base ** (dim / D))                         # (D/2,)
    # freqs 是递减的: 低维度频率高 (旋转快), 高维度频率低 (旋转慢)

    # ---- Step 3: 旋转角度 ----
    angles = pos * freqs  # (S, D/2)  广播: (S,1) * (D/2,)
    cos_a = torch.cos(angles)  # (S, D/2)
    sin_a = torch.sin(angles)  # (S, D/2)

    # ---- Step 4: 2D 旋转 ----
    # 把 D 维拆成 D/2 对，每对当作 2D 向量旋转
    # x[..., 0::2] 取第 0, 2, 4, ... 维 → (B, S, D/2)
    # x[..., 1::2] 取第 1, 3, 5, ... 维 → (B, S, D/2)
    def rotate(x):
        x1 = x[..., 0::2]  # (B, S, D/2)  偶数维
        x2 = x[..., 1::2]  # (B, S, D/2)  奇数维
        # 旋转: [x1', x2'] = [[cos,-sin],[sin,cos]] @ [x1, x2]
        r1 = x1 * cos_a - x2 * sin_a  # (B, S, D/2)
        r2 = x1 * sin_a + x2 * cos_a  # (B, S, D/2)
        # 交错合并回 D 维: stack → (B,S,D/2,2) → flatten → (B,S,D)
        return torch.stack([r1, r2], dim=-1).flatten(-2)

    return rotate(q), rotate(k)

In [ ]:
# Verify: compare scale=1 vs scale=4 frequencies, show norm preservation
B, S, D = 1, 8, 16
q = torch.randn(B, S, D)
k = torch.randn(B, S, D)

q1, k1 = ntk_rope(q, k, scale=1.0)
q4, k4 = ntk_rope(q, k, scale=4.0)

print("scale=1 base:", 10000.0 * (1.0 ** (D / (D - 2))))
print("scale=4 base:", 10000.0 * (4.0 ** (D / (D - 2))))
print()
print("Norm preservation (scale=1):")
print("  q input norm:", q.norm(dim=-1).mean().item())
print("  q output norm:", q1.norm(dim=-1).mean().item())
print()
print("Norm preservation (scale=4):")
print("  q input norm:", q.norm(dim=-1).mean().item())
print("  q output norm:", q4.norm(dim=-1).mean().item())

In [ ]:
from torch_judge import check
check("ntk_rope")

## 📝 核心思路总结

1. **RoPE 的本质**：把每对相邻维度当作 2D 向量做旋转，旋转角度由位置和频率共同决定，从而编码相对位置信息。
2. **NTK-aware 缩放只改 base**：`base_new = 10000 * scale^(D/(D-2))`，不改变旋转公式本身，通过调低频率来支持更长上下文。
3. **频率的多尺度特性**：低维度频率高（旋转快，编码局部位置），高维度频率低（旋转慢，编码全局位置），NTK 缩放主要拉伸低频维度。
4. **旋转是正交变换**：`[x1*cos - x2*sin, x1*sin + x2*cos]` 保持向量范数不变，这是 RoPE 能保持注意力分数相对性的关键。